# Machine Learning Module - Group Project Assignment
## Project Title: Universal Business Review Analyzer & Sentiment Intelligence System
### Domain: Multi-Industry Customer Feedback Analytics (Retail, Hospitality, Tech, Services)
### Sentiment Classes: 3-Class Classification (Negative, Neutral, Positive)

---
### Assignment Compliance Matrix
| Assignment Requirement | Implementation in this Notebook |
| :--- | :--- |
| **1. Problem Selection** | Multi-domain business review analyzer solving 3-class sentiment classification (Negative, Neutral, Positive). |
| **2. 100,000 Dataset & EDA** | Large-scale 100,000 record multi-domain dataset covering Retail, Hospitality, Consumer Tech, and Services. |
| **3. Data Visualizer (EDA Plots)** | Distribution of sentiment classes, domain breakdowns, word count distributions, keyword frequencies, correlation heatmaps, model comparisons, and 3x3 confusion matrices. |
| **4. Feature Engineering (Mandatory)** | **6 Meaningful Techniques**: Text cleaning, N-gram TF-IDF, metadata features, emotional punctuation signals, lexicon polarity, standard scaling. |
| **5. Multi-Model Development** | Comparison across candidate models including Logistic Regression, LinearSVC, and Random Forest Classifier. |
| **6. Model Evaluation** | Accuracy, Precision, Recall, F1-Score, 5-Fold Cross Validation, 3x3 Confusion Matrix, Classification Report. |
| **7. Model Serialization** | Complete 3-class pipeline bundle exported to `.joblib` for backend REST API deployment. |

## Step 1: Environment and Dependencies Setup
Initialize required libraries for Data Visualization, NLP, Feature Engineering, Machine Learning, and Model Serialization.

In [ ]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
os.makedirs("models", exist_ok=True)
print("[INFO] Environment initialized successfully with Data Visualizer suite.")

## Step 2: 100,000-Record Multi-Domain Dataset Ingestion & Quality Analysis
Load the 100,000 dataset containing 3 sentiment classes (`Negative = 0`, `Neutral = 1`, `Positive = 2`).

In [ ]:
data_path = "data/universal_business_reviews_100k.csv"

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}.")

df_full = pd.read_csv(data_path)

print("=" * 60)
print(f"Total Dataset Records : {len(df_full):,}")
print(f"Feature Columns       : {list(df_full.columns)}")
print(f"Missing Values Count  :\n{df_full.isnull().sum()}")
print(f"Duplicate Reviews     : {df_full.duplicated(subset=['review_text']).sum()}")
print("=" * 60)

df_clean = df_full.dropna(subset=['review_text']).reset_index(drop=True)

# Sample stratified working set for fast interactive notebook development
SAMPLE_SIZE = 30000
df, _ = train_test_split(
    df_clean, train_size=SAMPLE_SIZE, random_state=RANDOM_STATE, stratify=df_clean['sentiment_label']
)
df = df.reset_index(drop=True)

print(f"[INFO] Cleaned Stratified Working Sample: {len(df):,} records.")
print("\n3-Class Sentiment Distribution in Working Sample:")
print(df['sentiment_label'].value_counts(normalize=True).apply(lambda x: f"{x:.2%}"))

df.head()

## Step 3: Exploratory Data Visualizer (EDA Visualizations)
Visual analysis of sentiment class balance, business domain distribution, text length histograms, and keyword frequencies.

In [ ]:
# Visualizer 1: Sentiment Class Balance & Multi-Domain Donut Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {'Positive': '#2ecc71', 'Neutral': '#f39c12', 'Negative': '#e74c3c'}
sns.countplot(data=df, x='sentiment_label', hue='sentiment_label', order=['Positive', 'Neutral', 'Negative'], palette=palette, legend=False, ax=axes[0])
axes[0].set_title("Figure 1: Sentiment Class Balance in Working Sample", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Sentiment Category", fontsize=10)
axes[0].set_ylabel("Number of Reviews", fontsize=10)

for p in axes[0].patches:
    h = p.get_height()
    if h > 0:
        axes[0].annotate(f"{int(h):,}", (p.get_x() + p.get_width() / 2., h - (h * 0.1)),
                         ha='center', va='center', color='white', fontweight='bold', fontsize=11)

domain_counts = df['category'].value_counts()
axes[1].pie(domain_counts, labels=domain_counts.index, autopct='%1.1f%%', startangle=140,
            colors=['#3498db', '#9b59b6', '#e67e22', '#1abc9c'],
            wedgeprops=dict(width=0.4, edgecolor='w', linewidth=2))
axes[1].set_title("Figure 2: Multi-Domain Business Categories Breakdown", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualizer 2: Review Word Count & Character Length Distributions
df['char_length'] = df['review_text'].apply(len)
df['word_count_raw'] = df['review_text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='word_count_raw', hue='sentiment_label', palette=palette, kde=True, bins=30, ax=axes[0])
axes[0].set_title("Figure 3: Word Count Distribution by Sentiment Class", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Total Words in Review", fontsize=10)
axes[0].set_ylabel("Frequency", fontsize=10)

sns.boxplot(data=df, x='sentiment_label', y='char_length', hue='sentiment_label', order=['Positive', 'Neutral', 'Negative'], palette=palette, legend=False, ax=axes[1])
axes[1].set_title("Figure 4: Character Length Boxplot Across Sentiments", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Sentiment Category", fontsize=10)
axes[1].set_ylabel("Character Length", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizer 3: Cross-Domain Sentiment Breakdown & Distribution
fig, ax = plt.subplots(figsize=(12, 5))

domain_sentiment = pd.crosstab(df['category'], df['sentiment_label'], normalize='index') * 100
available_cols = [c for c in ['Positive', 'Neutral', 'Negative'] if c in domain_sentiment.columns]
domain_sentiment = domain_sentiment[available_cols]

domain_sentiment.plot(kind='bar', stacked=True, color=[palette[c] for c in available_cols], ax=ax, edgecolor='white')
ax.set_title("Figure 5: Sentiment Breakdown Across Business Categories (Percentage)", fontsize=12, fontweight='bold')
ax.set_xlabel("Business Category / Industry Domain", fontsize=10)
ax.set_ylabel("Percentage (%)", fontsize=10)
ax.legend(title="Sentiment Class", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)

for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', label_type='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualizer 4: Top Bi-Grams / Keyword Frequency in Positive vs Negative Reviews
def get_top_ngrams(corpus, n=2, top_k=10):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english', max_features=top_k).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return pd.DataFrame(words_freq, columns=['Ngram', 'Count'])

pos_reviews = df[df['sentiment_label'] == 'Positive']['review_text']
neg_reviews = df[df['sentiment_label'] == 'Negative']['review_text']

top_pos_df = get_top_ngrams(pos_reviews, n=2, top_k=10)
top_neg_df = get_top_ngrams(neg_reviews, n=2, top_k=10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=top_pos_df, y='Ngram', x='Count', color='#2ecc71', ax=axes[0])
axes[0].set_title("Figure 6A: Top Positive Bi-Grams", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Frequency Count", fontsize=10)

sns.barplot(data=top_neg_df, y='Ngram', x='Count', color='#e74c3c', ax=axes[1])
axes[1].set_title("Figure 6B: Top Negative Bi-Grams", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Frequency Count", fontsize=10)

plt.tight_layout()
plt.show()

## Step 4: Mandatory Feature Engineering (6 Techniques)
Section 5 of the assignment requires at least 5-6 meaningful feature engineering techniques:
1. **Technique 1: Text Preprocessing and Normalization** (Regex, lowercase, URL and special character removal)
2. **Technique 2: Text Length and Metadata Extraction** (`char_count`, `word_count`, `avg_word_len`)
3. **Technique 3: Emotional and Punctuation Signals** (`exclamation_count`, `uppercase_ratio`)
4. **Technique 4: Lexicon Sentiment Polarity Scoring** (Domain-Agnostic Polarity Index)
5. **Technique 5: Standard Scaling on Numerical Features** (`StandardScaler`)
6. **Technique 6: TF-IDF N-grams (1,2) with Sparse-Dense Fusion** (`scipy.sparse.hstack`)

In [ ]:
# TECHNIQUE 1: Text Preprocessing & Cleaning
NEGATION_TOKENS = set(['not', 'no', 'never', 'n\'t', 'hardly', 'barely', 'scarcely', 'without', 'lack', 'lacked', 'lacks', 'neither', 'nor'])

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can't", "can not", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"[^a-zA-Z\s!?'\.]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['review_text'].apply(clean_text)

# TECHNIQUE 2: Text Length & Metadata Features
df['char_count'] = df['cleaned_text'].apply(len)
df['word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))
df['avg_word_len'] = df['char_count'] / (df['word_count'] + 1e-5)

# TECHNIQUE 3: Emotional & Punctuation Signal Features
df['exclamation_count'] = df['review_text'].apply(lambda x: str(x).count('!'))
df['uppercase_ratio'] = df['review_text'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()) / (len(str(x)) + 1e-5)
)

# TECHNIQUE 4: Lexicon-Based Sentiment Polarity Scoring with Negation Window
POS_WORDS = set([
    'good', 'great', 'excellent', 'amazing', 'perfect', 'love', 'loved', 'best', 'fantastic',
    'superb', 'beautiful', 'clean', 'fast', 'fresh', 'tasty', 'delicious', 'sturdy',
    'smooth', 'recommend', 'recommended', 'wonderful', 'happy', 'pleased', 'flawless', 'premium',
    'kind', 'warm', 'cozy', 'elegant', 'comfortable', 'durable', 'stunning', 'fit',
    'fits', 'attentive', 'snappy', 'crisp', 'tender', 'juicy', 'punctual', 'honest',
    'pleasant', 'helpful', 'welcoming', 'delighted', 'impressed', 'impressive', 'miraculous',
    'exquisite', 'expert', 'luxurious', 'gourmet', 'dependable', 'trustworthy', 'outstanding',
    'enjoy', 'enjoyed', 'polite', 'prepared', 'organized', 'satisfactory', 'friendly',
    'professional', 'reasonable', 'flavor', 'flavorful', 'appreciation', 'valuable', 'exceptional'
])
NEG_WORDS = set([
    'bad', 'terrible', 'horrible', 'worst', 'poor', 'waste', 'broken', 'cheap',
    'defective', 'slow', 'rude', 'dirty', 'bland', 'overcooked', 'disappointed',
    'disappointing', 'awful', 'ripped', 'faded', 'unusable', 'crashed', 'disconnects', 'useless',
    'cold', 'ignored', 'stain', 'jammed', 'dropouts', 'drains', 'poisoning',
    'scam', 'shoddy', 'unprofessional', 'incompetent', 'frustrating', 'counterfeit',
    'knockoff', 'knock-off', 'fake', 'fraud', 'hazard', 'hazardous', 'overheated',
    'furious', 'refused', 'extortionate', 'garbage', 'damage', 'damaged', 'overcrowded',
    'unhelpful', 'unresponsive', 'undercooked', 'flavorless', 'lacked', 'fail', 'failed',
    'annoying', 'complaint', 'delay', 'delayed', 'unacceptable', 'regret', 'stale',
    'mediocre', 'subpar', 'inferior', 'dreadful', 'pathetic', 'gross', 'overpriced', 'high'
])

def compute_lexicon_polarity(text: str) -> float:
    words = clean_text(text).split()
    if not words:
        return 0.0
    pos_score = 0.0
    neg_score = 0.0
    negated = False
    negation_window = 0
    for w in words:
        if w in NEGATION_TOKENS:
            negated = True
            negation_window = 3
            continue
        if negation_window > 0:
            negation_window -= 1
            if negation_window == 0:
                negated = False
        if w in POS_WORDS:
            if negated:
                neg_score += 1.3
            else:
                pos_score += 1.0
        elif w in NEG_WORDS:
            if negated:
                pos_score += 0.4
            else:
                neg_score += 1.0
    total = pos_score + neg_score
    if total == 0:
        return 0.0
    return (pos_score - neg_score) / (total + 1.0)

df['lexicon_polarity'] = df['cleaned_text'].apply(compute_lexicon_polarity)

# TECHNIQUE 5: Feature Scaling on Numerical Columns
numeric_cols = ['char_count', 'word_count', 'avg_word_len', 'exclamation_count', 'uppercase_ratio', 'lexicon_polarity']
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(df[numeric_cols])

# TECHNIQUE 6: TF-IDF N-grams (1,2) Vectorization and Matrix Fusion
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True,
    min_df=2
)
X_text_tfidf = vectorizer.fit_transform(df['cleaned_text'])
X_combined = hstack([X_text_tfidf, csr_matrix(X_numeric_scaled)])
y = df['sentiment'].values

print(f"[SUCCESS] 6 Feature Engineering Techniques Applied. Feature Matrix Shape: {X_combined.shape}")

In [ ]:
# Visualizer 5: Feature Correlation Heatmap & Lexicon Polarity Distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Correlation Heatmap
corr_df = df[numeric_cols + ['sentiment']].corr()
sns.heatmap(corr_df, annot=True, cmap='coolwarm', fmt='.2f', linewidths=1, ax=axes[0], cbar_kws={'label': 'Correlation'})
axes[0].set_title("Figure 7: Feature Correlation Matrix Heatmap", fontsize=12, fontweight='bold')

# Polarity Distribution by Class
sns.kdeplot(data=df, x='lexicon_polarity', hue='sentiment_label', palette=palette, fill=True, common_norm=False, ax=axes[1])
axes[1].set_title("Figure 8: Lexicon Polarity Index Separation", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Lexicon Sentiment Polarity Score (-1.0 to +1.0)", fontsize=10)
axes[1].set_ylabel("Density", fontsize=10)

plt.tight_layout()
plt.show()

## Step 5: Multi-Model Development and Benchmarking
Train and evaluate candidate machine learning models for 3-class classification (`0: Negative`, `1: Neutral`, `2: Positive`).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training set size: {X_train.shape[0]:,} | Testing set size: {X_test.shape[0]:,}")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=2.0, random_state=RANDOM_STATE),
    "Linear Support Vector (LinearSVC)": LinearSVC(C=1.0, dual='auto', random_state=RANDOM_STATE, max_iter=2000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=50, max_depth=15, n_jobs=-1, random_state=RANDOM_STATE)
}

results = []
trained_models = {}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    trained_models[name] = clf
    y_pred = clf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cv_score = cross_val_score(clf, X_train, y_train, cv=5, scoring='f1_weighted', n_jobs=-1).mean()
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "5-Fold CV F1": cv_score
    })

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
results_df

In [ ]:
# Visualizer 6: Multi-Model Benchmark Comparison Plot
fig, ax = plt.subplots(figsize=(10, 5))

df_plot = results_df.melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score', '5-Fold CV F1'],
                          var_name='Metric', value_name='Score')

sns.barplot(data=df_plot, x='Model', y='Score', hue='Metric', palette='viridis', ax=ax)
ax.set_title("Figure 9: Candidate Multi-Model Performance Benchmarking Comparison", fontsize=12, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_ylabel("Evaluation Score (0.0 - 1.0)", fontsize=10)
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=10)

for p in ax.patches:
    h = p.get_height()
    if h > 0.5:
        ax.annotate(f"{h:.3f}", (p.get_x() + p.get_width() / 2., h + 0.01),
                    ha='center', va='bottom', fontsize=8, rotation=90)

plt.tight_layout()
plt.show()

## Step 6: 3-Class Model Evaluation and Confusion Matrix Visualization

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
y_best_pred = best_model.predict(X_test)

print(f"Selected Champion Model: {best_model_name}")
print("\n--- Detailed 3-Class Classification Report ---")
print(classification_report(y_test, y_best_pred, target_names=['Negative (0)', 'Neutral (1)', 'Positive (2)']))

# Visualizer 7: Raw and Normalized 3x3 Confusion Matrix Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_raw = confusion_matrix(y_test, y_best_pred)
cm_norm = confusion_matrix(y_test, y_best_pred, normalize='true')
labels = ['Negative', 'Neutral', 'Positive']

sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title(f"Figure 10A: 3x3 Confusion Matrix (Raw Counts) - {best_model_name}", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Predicted Sentiment", fontsize=10)
axes[0].set_ylabel("True Sentiment", fontsize=10)

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title(f"Figure 10B: 3x3 Confusion Matrix (Normalized %) - {best_model_name}", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Predicted Sentiment", fontsize=10)
axes[1].set_ylabel("True Sentiment", fontsize=10)

plt.tight_layout()
plt.show()

## Step 7: Pipeline Serialization & Artifact Export
Export the complete 3-class pipeline bundle for FastAPI backend integration.

In [ ]:
pipeline_bundle = {
    "model": best_model,
    "vectorizer": vectorizer,
    "scaler": scaler,
    "numeric_features": numeric_cols,
    "positive_lexicon": list(POS_WORDS),
    "negative_lexicon": list(NEG_WORDS),
    "class_mapping": {0: "Negative", 1: "Neutral", 2: "Positive"},
    "best_model_name": best_model_name
}

joblib_path = "models/business_sentiment_pipeline.joblib"
joblib.dump(pipeline_bundle, joblib_path)
print(f"[SAVED] Pipeline bundle saved to: {joblib_path}")

metadata = {
    "model_name": best_model_name,
    "num_classes": 3,
    "classes": ["Negative", "Neutral", "Positive"],
    "metrics": results_df.to_dict(orient="records"),
    "num_features": X_combined.shape[1],
    "sample_count": len(df)
}
with open("models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
print("[SAVED] Metadata saved to: models/model_metadata.json")

## Step 8: Multi-Industry Inference Testing

In [ ]:
def predict_sentiment(text: str, artifact_path=joblib_path):
    bundle = joblib.load(artifact_path)
    clf = bundle["model"]
    vec = bundle["vectorizer"]
    scl = bundle["scaler"]
    num_cols = bundle["numeric_features"]
    
    cleaned = clean_text(text)
    char_c = len(cleaned)
    word_c = len(cleaned.split())
    avg_w = char_c / (word_c + 1e-5)
    excl_c = str(text).count('!')
    upper_r = sum(1 for c in str(text) if c.isupper()) / (len(str(text)) + 1e-5)
    polarity = compute_lexicon_polarity(cleaned)
    
    num_df = pd.DataFrame([[char_c, word_c, avg_w, excl_c, upper_r, polarity]], columns=num_cols)
    num_scaled = scl.transform(num_df)
    tfidf_vec = vec.transform([cleaned])
    feat_matrix = hstack([tfidf_vec, csr_matrix(num_scaled)])
    
    pred_code = clf.predict(feat_matrix)[0]
    label = bundle["class_mapping"][pred_code]
    
    if hasattr(clf, "predict_proba"):
        confidence = clf.predict_proba(feat_matrix)[0][pred_code]
    else:
        confidence = 1.0
        
    return label, confidence

test_cases = [
    ("Hospitality & Food", "I had a pleasant experience at this restaurant. The food was delicious, fresh, and nicely presented. The staff were friendly, helpful, and professional. The atmosphere was comfortable and welcoming, making it a great place to enjoy a meal. I appreciated the service and would happily recommend this restaurant to friends and family."),
    ("Hospitality & Food", "Horrible service, cold food, and we waited forty minutes for our water."),
    ("Retail & Apparel", "The winter coat is cozy, elegant, and fits like a dream."),
    ("Retail & Apparel", "Cheap fabric, ripped seams after one wash, and customer support refused an exchange."),
    ("Consumer Electronics", "Outstanding battery life, crystal clear display, and fast charging speed."),
    ("General Neutral", "An ordinary experience regarding the product. It features acceptable quality and routine service.")
]

for domain, review in test_cases:
    sentiment, conf = predict_sentiment(review)
    badge = f"[{sentiment.upper()}]"
    print(f"Domain : {domain}")
    print(f"Review : \"{review}\"")
    print(f"Result : {badge} (Confidence: {conf:.2%})\n")